In [ ]:
# # Unsloth 설치 및 최신 버전 업그레이드
# !pip install unsloth
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install trl bitsandbytes triton xformers peft

# # GPU가 softcapping을 지원하는 경우, Flash Attention 2 설치
# import torch
# if torch.cuda.get_device_capability()[0] >= 8:
#     !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

# 1. 모델 로드

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None, # None 인경우 컴퓨팅 유닛에 알맞는 dtype으로 설정됨
    load_in_4bit = True, # 4bit 양자화
    token = "hf ..." # huggingface Token 입력
)

# 2. QLoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['p_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.1,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = {"quantization": "4bit"},
)

# 3. 데이터 셋 EOS 추가

In [ ]:
from datasets import load_dataset

EOS_TOKEN = tokenizer.eos_token
dataset = load_dataset("csv", data_files='datasets/formatted_finance.csv', split="train")

def add_eos_token(examples):
    examples['formatted_text'] = examples["formatted_text"] + EOS_TOKEN
    return examples

dataset = dataset.map(add_eos_token)


# 4. 모델 학습

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_schedule_type = 'cosine',
        seed = 42,
        
        # save 관련 파라미터
        output_dir = 'outputs',
        save_steps = 100,
        save_total_limit = 1,
    )
)

In [ ]:
trainer_stats = trainer.train()

# 5. 모델 추론

In [ ]:
# 금융 용어 학습 확인
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [
        """
        BIC에 대해서 알려줘.
        """
    ],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    """
    다음 중 화폐의 시간가치에 관한 설명으로 옳지 않은 것은 무엇인가?

A. 월 복리의 경우, 매월 적용되는 이자율은 연간 명목 이자율을 1/12로 나누어 산출한다.
B. 투자 원금 및 기타 조건이 동일할 경우, 단리 방식보다 복리 방식에서 발생하는 이자가 더 크다.
C. 일시불로 지급될 금액의 현재 가치는 미래 가치를 일정 기간 동안 할인율을 적용해 산출할 수 있다.
D. 1,000,000원을 연 5% 복리로 2년 동안 예치했을 경우, 만기에 받을 세전 이자는 100,000원이다.
    """
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    """
다음 표는 주식 A에 대한 분석 결과입니다. 표를 바탕으로 향후 A의 주가가 상승할지 하락할지 예측하고 [증가] 혹은 [감소] 라고 답변하시요.
### 분석:
|    | date       |   open |   high |   low |   close |   adj-close |   inc-5 |   inc-10 |   inc-15 |   inc-20 |   inc-25 |   inc-30 |
|---:|:-----------|-------:|-------:|------:|--------:|------------:|--------:|---------:|---------:|---------:|---------:|---------:|
|  0 | 2020-10-19 |    1.2 |    2.2 |  -0.6 |    -0.1 |        -0.1 |    -1.4 |     -1.2 |     -0.8 |     -2.4 |     -2.4 |     -2.6 |
|  1 | 2020-10-20 |    0.6 |    1.2 |  -0.6 |     0.1 |         0.1 |    -0.8 |     -0.8 |     -0.7 |     -2.1 |     -2.3 |     -2.5 |
|  2 | 2020-10-21 |    1.6 |    2.3 |  -0   |    -2   |        -2   |     1.3 |      1.2 |      1.3 |      0.3 |     -0.4 |     -0.5 |
|  3 | 2020-10-22 |   -2.6 |    0.3 |  -2.9 |     3.2 |         3.2 |    -1.3 |     -1.9 |     -1.8 |     -2.1 |     -3.5 |     -3.3 |
|  4 | 2020-10-23 |    1.6 |    1.7 |  -0.8 |    -1   |        -1   |    -0.3 |     -0.9 |     -0.8 |     -0.8 |     -2.3 |     -2.2 |
|  5 | 2020-10-26 |    2.6 |    2.9 |  -1.1 |    -3.9 |        -3.9 |     3   |      2.7 |      2.8 |      3.1 |      1.8 |      1.7 |
|  6 | 2020-10-27 |    2.5 |    3.2 |  -0.2 |    -3.5 |        -3.5 |     5.2 |      6   |      6.3 |      6.5 |      5.4 |      5.2 |
|  7 | 2020-10-28 |    3.3 |    3.9 |  -0.1 |    -4.6 |        -4.6 |     8.1 |     10.1 |     10.6 |     11   |     10.3 |      9.8 |
|  8 | 2020-10-29 |   -0.2 |    1.2 |  -1.1 |     0.1 |         0.1 |     5.2 |      8.9 |      9.6 |     10.3 |     10.3 |      9.3 |
|  9 | 2020-10-30 |    1.9 |    3.3 |  -1.9 |    -2.6 |        -2.6 |     4.8 |     10.2 |     11.5 |     12.4 |     13   |     11.8 |
    """
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    """
다음 문제를 읽고 정답으로 가장 알맞은 것을 고르시요.
### 질문: 엑세스바이오의 코로나 진단 제품이 주요 매출에 기여한 이유와, 해당 제품들이 미국 시장에서 어떻게 판매되고 있는지를 설명해 주세요.
### 선택지: 
A. 엑세스바이오의 코로나 진단 제품은 바이러스와 관련 없는 일반적인 건강 검진용으로 개발되었으며, 미국 시장에선 주로 소비자 직거래 방식으로 판매되고 있습니다.
B. 엑세스바이오의 코로나 진단 제품은 대규모 임상시험을 통해 효과가 입증되어, 기존 진단 키트의 80% 이상을 차지하고 있으며, 주로 소매점에서 판매되고 있습니다.
C. 엑세스바이오의 코로나 진단 제품은 에볼라 및 기타 질병 진단에 사용되고 있으며, 미국 시장에서는 주 정부 계약을 통해 독점 판매되고 있습니다.
D. 엑세스바이오의 코로나 진단 제품은 COVID-19의 급격한 확산에 대응하여 개발된 신속진단키트로, 주로 연간 매출액에서 99.5%를 차지하고 있습니다. 이 제품들은 미국 시장에서 의료기기 전문 유통사를 통해 판매되고 있으며, 미 FDA로부터 긴급사용승인을 획득한 덕분에 더욱 폭넓은 수요를 만나게 되었습니다.
    """
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    """
다음 문제를 읽고 정답으로 가장 알맞은 것을 고르시요.
### 질문: 다음 중 우리나라 주식시장 매매 제도에 대한 기술로 맞는 것은?
### 선택지: 
A. 개장 시간은 오전 10시다.
B. 유가증권시장의 가격 제한폭은 전일 종가 대비 상하 15%이다.
C. 코스닥시장에는 가격 제한폭이 없다.
D. 점심시간(12~1시)에는 휴장한다.
E. 동시호가는 폐장 시간에만 적용한다.
F. K-OTC시장의 가격 제한폭은 전일 종가 대비 상하 30%이다.
G. K-OTC시장의 운영시간은 09:00부터 16:00까지이다.
    """
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    """
주어진 데이터프레임을 보고 질문에 요구에 알맞는 코드를 선택하시요.
### df.head()
|    | Symbol    | Series   | Date        |   Prev Close |   Open Price |   High Price |   Low Price |   Last Price |   Close Price |   Average Price |   Total Traded Quantity |    Turnover |   No. of Trades |   Deliverable Qty |   % Dly Qt to Traded Qty |
|---:|:----------|:---------|:------------|-------------:|-------------:|-------------:|------------:|-------------:|--------------:|----------------:|------------------------:|------------:|----------------:|------------------:|-------------------------:|
|  0 | GODREJIND | EQ       | 15-May-2017 |       564.6  |       581    |       584    |      568.5  |       578.9  |        578.55 |          578.09 |                  797171 | 4.60836e+08 |           21649 |            360927 |                    45.28 |
|  1 | GODREJIND | EQ       | 16-May-2017 |       578.55 |       581.45 |       589    |      572.25 |       583.8  |        584.8  |          583.6  |                  500223 | 2.9193e+08  |           17204 |            210364 |                    42.05 |
|  2 | GODREJIND | EQ       | 17-May-2017 |       584.8  |       583    |       594    |      576.85 |       584.9  |        588.6  |          588.74 |                  504155 | 2.96815e+08 |            8567 |            261667 |                    51.9  |
|  3 | GODREJIND | EQ       | 18-May-2017 |       588.6  |       582    |       588.85 |      571.2  |       572.25 |        574.6  |          580.9  |                  223583 | 1.29879e+08 |            7144 |             99785 |                    44.63 |
|  4 | GODREJIND | EQ       | 19-May-2017 |       574.6  |       581    |       585.8  |      567.55 |       579.85 |        578    |          577.31 |                  245436 | 1.41692e+08 |            4969 |             68041 |                    27.72 |

### 질문:
"종가" 열의 평균 값을 계산합니다.

### 선택지:
a) ```python
df['Close Price'].mean()
```
b) ```python
df['Close_Price'].mean()
```
c) ```python
df['Total Traded Quantity'].median()
```
d) ```python
sum(df['Close Price']) / len(df['Close_Price'])
```
    """
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

# 6. 모델 저장 및 업로드

In [ ]:
model_name = ""

# LoRA Adapter 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Merged model 저장 및 업로드
model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged(model_name, tokenizer, save_method = "merged_16bit", token = "hf_...") # huggingface token 입력해 주세요!!